# Persian Character Recognition — saved preparation and training

Prepare the deterministic images once, save them to Drive, and reuse them for later training sessions. The default `DATA_ACTION = 'reuse'` reads the existing verified preparation snapshot. It does not repeat the archive audit or font generation. On its first use it creates a prepared image cache; later sessions restore that cache.

The current archive supplies 32 handwritten labels; printed data supplies all 33 labels including `آ`. Handwritten `آ` remains unvalidated and independent Persian-reader review is pending.

## Exactly which blocks to run

The code cells are labeled `BLOCK 1` through `BLOCK 13`. Reload this saved notebook before following these instructions.

| Task | Settings in Block 1 | Run in order | Expected Block 11 message |
| --- | --- | --- | --- |
| First time: build the permanent prepared-image cache without training | `EXPERIMENT_MODE = 'prepare'`, `DATA_ACTION = 'reuse'`, both training toggles `False` | Blocks **1–11** | **BUILT** |
| Images already cached; load them without training | Same preparation settings | Blocks **1–11** | **REUSED** |
| Images already cached; train in a fresh/reconnected runtime | `EXPERIMENT_MODE = 'prototype'`, `DATA_ACTION = 'reuse'`, `RUN_PROTOTYPE_TRAINING = True`, `RUN_FINAL_TRAINING = False` | Blocks **1–13** | **REUSED**, then training starts in Block 13 |

Keep `DATA_ACTION = 'reuse'` for both first-time cache creation and later loading. Block 11 automatically builds the cache if missing; otherwise it loads the saved PNGs. Blocks 3–10 load manifests/checks and define functions; their image audit, font generation, and preprocessing previews are skipped in reuse mode. Running these cells again initializes the new Python runtime without repeating image preparation.

`DATA_ACTION = 'prepare_new'` is reserved for deliberately making a new data/preprocessing version. Switching `EXPERIMENT_MODE` from `prepare` to `prototype` creates a new model run, so execute Blocks 1–13 in order after that switch. Block 13 starts a new training run; it does not resume an interrupted optimizer/checkpoint.

Saved data lives under `artifacts/prepared/<cache_id>/` in `MyDrive/Persian Character Recognition`. The PNGs are packed in `prepared_images.zip` for a single Drive transfer, then extracted to a local `persian_prepared/<cache_id>/images/` folder for training. Random augmentation and tensor normalization remain in the training loader.

Required project layout for a new preparation is `04_final.zip`, `config/characters.json`, the frozen audit, and `fonts/<family>/*.ttf`. Reusing a complete cache does not open the raw archive or font files. Training results get their own timestamped folder, independently of the saved dataset.

In [ ]:
# BLOCK 1 — Settings and mount Google Drive
from pathlib import Path
import os, shutil, subprocess, sys, hashlib, json, random, math, io, time, zipfile, tempfile, re
from datetime import datetime, timezone

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Local runtime; PERSIAN_* path overrides are supported.')

PROJECT_DIR = Path('/content/drive/MyDrive/Persian Character Recognition') if IN_COLAB else Path(os.environ.get('PERSIAN_PROJECT_DIR', Path.cwd()))
ARCHIVE_DRIVE = Path(os.environ.get('PERSIAN_ARCHIVE_PATH', PROJECT_DIR / '04_final.zip'))
CHARACTER_MAP = PROJECT_DIR / 'config' / 'characters.json'
FONT_DIR = Path(os.environ.get('PERSIAN_FONT_DIR', PROJECT_DIR / 'fonts'))
ARTIFACT_ROOT = Path(os.environ.get('PERSIAN_ARTIFACT_ROOT', PROJECT_DIR / 'artifacts'))
NOTEBOOK_PATH = PROJECT_DIR / 'notebooks' / '01_data_preparation_training.ipynb'

EXPERIMENT_MODE = 'prepare'  # prepare | prototype | final
RUN_PROTOTYPE_TRAINING = False
RUN_FINAL_TRAINING = False
SEED = 42  # model/subset seed; never changes frozen source-form or font partitions
if EXPERIMENT_MODE != 'final' and SEED != 42:
    raise ValueError('The current prototype budget uses seed 42; final-study seeds are configured separately.')
DATA_ACTION = 'reuse'  # reuse | prepare_new
PREPARATION_SOURCE_DIR = Path(os.environ.get('PERSIAN_PREPARATION_SOURCE_DIR', ARTIFACT_ROOT / 'phase3' / 'prototype_seed_42_20261008T153636952896Z'))
CACHE_ROOT = ARTIFACT_ROOT / 'prepared'
RUNTIME_CACHE_ROOT = Path(os.environ.get('PERSIAN_RUNTIME_CACHE_DIR', Path(tempfile.gettempdir()) / 'persian_prepared'))
EXPECTED_ARCHIVE_SHA256 = '1faaff1234e743897652ab41e4bf3c3a2c2d87fdf1eda7886516f9bf031786e9'
EXPECTED_FROZEN_MANIFEST_SHA256 = '04187fc486de34919e9e31230f7a38ce33cf1aad29fdcaaa7a15cafdd45ff95f'
LOCAL_ARCHIVE = Path(tempfile.gettempdir()) / f'04_final_{EXPECTED_ARCHIVE_SHA256[:12]}.zip'

if DATA_ACTION not in {'reuse', 'prepare_new'}:
    raise ValueError('DATA_ACTION must be reuse or prepare_new.')
USE_SAVED_PREPARATION = DATA_ACTION == 'reuse'
RUN_PHASE = {'prepare': 'phase2', 'prototype': 'phase3', 'final': 'phase4'}.get(EXPERIMENT_MODE)
if RUN_PHASE is None:
    raise ValueError('EXPERIMENT_MODE must be prepare, prototype, or final.')
signature = (str(ARTIFACT_ROOT), EXPERIMENT_MODE, SEED, DATA_ACTION, str(PREPARATION_SOURCE_DIR))
if globals().get('_RUN_SIGNATURE') != signature or not globals().get('RUN_DIR', Path('/nonexistent')).exists():
    for stale_name in ['PROTOTYPE_READY', 'FINAL_READY', 'PREPARED_DATA_READY', 'PREPARED_RUN_ID', 'cached_prototype_manifest', 'cache_info']:
        globals().pop(stale_name, None)
    RUN_ID = EXPERIMENT_MODE + '_seed_' + str(SEED) + '_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    RUN_DIR = ARTIFACT_ROOT / RUN_PHASE / RUN_ID
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    _RUN_SIGNATURE = signature

def ensure_local_archive():
    # Lazy: a complete prepared-cache hit never opens/copies the raw archive.
    if LOCAL_ARCHIVE.exists() and sha256_file(LOCAL_ARCHIVE) == EXPECTED_ARCHIVE_SHA256:
        return LOCAL_ARCHIVE
    if not ARCHIVE_DRIVE.exists():
        raise FileNotFoundError(f'First cache build needs the pinned source archive: {ARCHIVE_DRIVE}')
    pending = LOCAL_ARCHIVE.with_suffix('.copying')
    shutil.copy2(ARCHIVE_DRIVE, pending)
    if sha256_file(pending) != EXPECTED_ARCHIVE_SHA256:
        pending.unlink()
        raise ValueError('The archive differs from the frozen Phase 1 input; a new audit/data version is required.')
    pending.replace(LOCAL_ARCHIVE)
    return LOCAL_ARCHIVE

if not CHARACTER_MAP.exists():
    raise FileNotFoundError(f'Missing label mapping: {CHARACTER_MAP}')
print('Project:', PROJECT_DIR)
print('Data action:', DATA_ACTION, '| preparation snapshot:', PREPARATION_SOURCE_DIR)
print('Training/output run:', RUN_DIR)
random.seed(SEED)

In [ ]:
# BLOCK 2 — Load libraries
# Data-preparation dependencies only. PyTorch is loaded in the model cell for prototype/final runs.
try:
    import numpy as np
    import pandas as pd
    from PIL import Image, ImageOps, ImageDraw, ImageFont, features
except ImportError as exc:
    raise RuntimeError('Use Colab or install numpy, pandas, and Pillow in the local environment.') from exc
try:
    from IPython.display import display
except ImportError:
    display = lambda image: None  # Preview PNGs are still saved to the run folder.

print('NumPy:', np.__version__, '| pandas:', pd.__version__, '| Pillow:', Image.__version__)
print('Pillow RAQM shaping available:', features.check_feature('raqm'))
print('Mode:', EXPERIMENT_MODE, '| seed:', SEED)

## 1. Load the frozen preparation snapshot

The default path reads small metadata and manifests from the existing completed preparation. It preserves the audited source-form partitions and saved font groups. No image audit, archive copying, or font rendering runs on a cache hit. A changed dataset or preprocessing method requires a new preparation snapshot.

In [ ]:
# BLOCK 3 — Load the saved preparation snapshot
labels = json.loads(CHARACTER_MAP.read_text(encoding='utf-8'))
FROZEN_AUDIT_DIR = PREPARATION_SOURCE_DIR / 'audit'
if not FROZEN_AUDIT_DIR.exists():
    FROZEN_AUDIT_DIR = ARTIFACT_ROOT / 'phase1'
for name in ['audit.json', 'letter_manifest.csv']:
    if not (FROZEN_AUDIT_DIR / name).exists():
        raise FileNotFoundError(f'Missing frozen audit file: {FROZEN_AUDIT_DIR / name}. Restore the audited preparation snapshot.')
if sha256_file(FROZEN_AUDIT_DIR / 'letter_manifest.csv') != EXPECTED_FROZEN_MANIFEST_SHA256:
    raise ValueError('The source-form manifest differs from the frozen Phase 1 assignments.')
audit = json.loads((FROZEN_AUDIT_DIR / 'audit.json').read_text(encoding='utf-8'))
assert audit['archive_sha256'] == EXPECTED_ARCHIVE_SHA256
assert audit['decode_failures'] == []
frozen_handwriting = pd.read_csv(FROZEN_AUDIT_DIR / 'letter_manifest.csv')

if USE_SAVED_PREPARATION:
    required_snapshot_files = ['training_manifest.csv', 'run_config.json', 'preprocessing.json',
        'preprocessing_source.py', 'font_splits.json', 'font_render_check.json']
    missing = [name for name in required_snapshot_files if not (PREPARATION_SOURCE_DIR / name).exists()]
    if missing:
        raise FileNotFoundError(f'Preparation snapshot is incomplete: {missing}. Set PREPARATION_SOURCE_DIR to the completed preparation folder.')
    source_config = json.loads((PREPARATION_SOURCE_DIR / 'run_config.json').read_text(encoding='utf-8'))
    SOURCE_MANIFEST_SHA256 = sha256_file(PREPARATION_SOURCE_DIR / 'training_manifest.csv')
    assert source_config['manifest_sha256']['training_manifest.csv'] == SOURCE_MANIFEST_SHA256, 'Preparation manifest changed.'
    assert source_config['archive_sha256'] == EXPECTED_ARCHIVE_SHA256
    assert source_config['class_order'] == labels['class_order']
    snapshot_preprocessing = json.loads((PREPARATION_SOURCE_DIR / 'preprocessing.json').read_text(encoding='utf-8'))
    snapshot_preprocessing_source = (PREPARATION_SOURCE_DIR / 'preprocessing_source.py').read_text(encoding='utf-8')
    assert hashlib.sha256(snapshot_preprocessing_source.encode()).hexdigest() == snapshot_preprocessing['source_sha256']
    assert snapshot_preprocessing == source_config['preprocessing']
    font_splits = json.loads((PREPARATION_SOURCE_DIR / 'font_splits.json').read_text(encoding='utf-8'))
    assert font_splits == source_config['font_groups']
    assert sha256_file(PREPARATION_SOURCE_DIR / 'font_render_check.json') == source_config['font_render_check_sha256']
    saved_manifest = pd.read_csv(PREPARATION_SOURCE_DIR / 'training_manifest.csv', keep_default_na=False)
    archive_mask = saved_manifest['archive_member'].astype(str).ne('')
    handwriting = saved_manifest.loc[archive_mask].copy()
    frozen_identity = frozen_handwriting.set_index('archive_member')[['split', 'source_group']].sort_index().astype({'split': str, 'source_group': str})
    saved_identity = handwriting.set_index('archive_member')[['split', 'source_group']].sort_index().astype({'split': str, 'source_group': str})
    pd.testing.assert_frame_equal(saved_identity, frozen_identity)
    printed = saved_manifest[saved_manifest['source_type'] == 'printed'].copy()
    handwritten_aa = saved_manifest[(saved_manifest['source_type'] == 'handwritten') & ~archive_mask].copy()
    font_render_checks = json.loads((PREPARATION_SOURCE_DIR / 'font_render_check.json').read_text(encoding='utf-8'))
    if isinstance(font_render_checks, dict):
        font_render_checks = font_render_checks.get('checks', font_render_checks.get('font_checks', []))
    AUDIT_OUT = FROZEN_AUDIT_DIR
    print('Loaded saved preparation. Archive audit and font generation skipped.')
else:
    ensure_local_archive()
    AUDIT_OUT = RUN_DIR / 'audit'
    AUDIT_OUT.mkdir(parents=True, exist_ok=True)
    for name in ['audit.json', 'letter_manifest.csv', 'handwriting_samples.png']:
        if (FROZEN_AUDIT_DIR / name).exists():
            shutil.copy2(FROZEN_AUDIT_DIR / name, AUDIT_OUT / name)
    handwriting = frozen_handwriting.copy()
    source_config = {}
    print('Creating a new printed/manual preparation using the frozen archive manifest; the audit is reused.')
print('Frozen archive:', EXPECTED_ARCHIVE_SHA256, '| unique handwriting rows:', len(handwriting))

In [ ]:
# BLOCK 4 — Check the 33-label order
assert len(labels['class_order']) == 33 and labels['class_order'][-1] == 'آ'
target_index = {character: i for i, character in enumerate(labels['class_order'])}
source_map = {int(c['source_folder']): c['character'] for c in labels['classes'] if c['source_folder'] is not None}
assert source_map == {10 + i: labels['class_order'][i] for i in range(32)}
if not USE_SAVED_PREPARATION:
    handwriting['character'] = handwriting['source_folder'].astype(int).map(source_map)
    handwriting['target_idx'] = handwriting['character'].map(target_index)
    handwriting['source_type'] = 'handwritten'
    handwriting['is_prepared'] = False
    assert handwriting['character'].notna().all() and handwriting['target_idx'].notna().all()
display(handwriting.groupby(['split', 'character']).size().unstack(fill_value=0).iloc[:, :8])
print('Archive coverage:', sorted(handwriting['character'].unique()))
print('No separate archive class for:', sorted(set(labels['class_order']) - set(handwriting['character'].unique())))

## 2. Inspect archive labels and sample quality

This training-only contact sheet supports a visual spot check. Independent Persian-reader review remains a separate status and is not set to approved by running the notebook.

In [ ]:
# BLOCK 5 — Reader-review status
LABEL_REVIEWER = ''
LABEL_REVIEW_APPROVED = False
if not USE_SAVED_PREPARATION and (AUDIT_OUT / 'handwriting_samples.png').exists():
    display(Image.open(AUDIT_OUT / 'handwriting_samples.png'))
print('Reader review:', 'approved' if LABEL_REVIEW_APPROVED and LABEL_REVIEWER.strip() else 'pending; labels remain provisional')

## 3. Shared preparation definition

This cell defines the deterministic preparation used to build a cache and to process future uploads. A saved cache is checked against its exact source/settings. Only a cache miss prepares raw images; training loads the saved 224×224 grayscale PNGs. Random affine/blur augmentation and normalization still happen per training batch.

In [ ]:
# BLOCK 6 — Define and verify preprocessing (does not process images in reuse mode)
OUTPUT_SIZE = 224
CONTENT_SIZE = 192

def prepare_image(image, output_size=OUTPUT_SIZE, content_size=CONTENT_SIZE):
    image = ImageOps.exif_transpose(image)
    if 'A' in image.getbands() or image.mode == 'P':
        rgba = image.convert('RGBA')
        white = Image.new('RGBA', rgba.size, (255, 255, 255, 255))
        image = Image.alpha_composite(white, rgba).convert('RGB')
    gray = image.convert('L')
    pixels = np.asarray(gray)
    border = np.concatenate([pixels[0, :], pixels[-1, :], pixels[:, 0], pixels[:, -1]])
    if float(np.median(border)) < 127.5:
        gray = ImageOps.invert(gray)
    scale = min(content_size / gray.width, content_size / gray.height)
    resized_size = (max(1, round(gray.width * scale)), max(1, round(gray.height * scale)))
    gray = gray.resize(resized_size, Image.Resampling.BILINEAR)
    canvas = Image.new('L', (output_size, output_size), 255)
    canvas.paste(gray, ((output_size - gray.width) // 2, (output_size - gray.height) // 2))
    return canvas

def show_preprocessing_examples(n=8):
    rows = handwriting[handwriting['split'] == 'train'].sample(n=min(n, len(handwriting)), random_state=SEED)
    sheet = Image.new('RGB', (640, max(1, len(rows)) * 150), 'white')
    draw = ImageDraw.Draw(sheet)
    with zipfile.ZipFile(LOCAL_ARCHIVE) as zf:
        for i, (_, row) in enumerate(rows.iterrows()):
            raw = Image.open(io.BytesIO(zf.read(row['archive_member']))).convert('L')
            prepared = prepare_image(raw)
            sheet.paste(raw.convert('RGB').resize((112, 112)), (8, i * 150 + 24))
            sheet.paste(prepared.convert('RGB').resize((112, 112)), (145, i * 150 + 24))
            draw.text((8, i * 150 + 5), f"Source {row['character']}", fill='black')
            draw.text((145, i * 150 + 5), 'Prepared once', fill='black')
    preview_path = RUN_DIR / 'handwriting_preprocessing_preview.png'
    sheet.save(preview_path)
    display(sheet)
    print('Saved:', preview_path)

if not USE_SAVED_PREPARATION:
    show_preprocessing_examples()
PREPROCESSING = {'version': 'gray-pad-v1', 'output_size': OUTPUT_SIZE, 'content_size': CONTENT_SIZE,
                 'orientation': 'EXIF transpose', 'transparency': 'composite on white',
                 'background': 'infer from median border; normalize to light',
                 'resize': 'aspect-preserving bilinear; center on white canvas; no crop',
                 'input_color': 'grayscale repeated to RGB', 'mean': [0.485, 0.456, 0.406],
                 'std': [0.229, 0.224, 0.225],
                 'printed_rendering': 'RAQM glyph bounding box plus proportional white margin; shared preparation once',
                 'printed_margin_fraction_by_size': {'24': 0.55, '54': 0.20}}
def extract_preprocessing_source(notebook_path):
    if not notebook_path.exists():
        raise FileNotFoundError(f'Notebook source required for the reproducible model bundle: {notebook_path}')
    notebook = json.loads(notebook_path.read_text(encoding='utf-8'))
    for cell in notebook.get('cells', []):
        source = ''.join(cell.get('source', []))
        start = source.find('def prepare_image(')
        if start >= 0:
            end = source.find('\ndef show_preprocessing_examples', start)
            if end < 0:
                raise ValueError('Could not find the end of prepare_image in the saved notebook.')
            return source[start:end].rstrip() + '\n'
    raise ValueError('Could not find prepare_image in the saved notebook.')

PREPROCESSING_SOURCE = extract_preprocessing_source(NOTEBOOK_PATH)
reference_namespace = {'OUTPUT_SIZE': OUTPUT_SIZE, 'CONTENT_SIZE': CONTENT_SIZE}
exec(compile(PREPROCESSING_SOURCE, '<saved-preparation-source>', 'exec'), reference_namespace)
reference_prepare = reference_namespace['prepare_image']
code_fields = ['co_code', 'co_consts', 'co_names', 'co_varnames', 'co_argcount', 'co_kwonlyargcount', 'co_freevars']
if prepare_image.__defaults__ != reference_prepare.__defaults__ or any(getattr(prepare_image.__code__, field) != getattr(reference_prepare.__code__, field) for field in code_fields):
    raise RuntimeError('The executing preparation differs from the saved source. Save the notebook and create a new preparation version.')
PREPROCESSING['source_sha256'] = hashlib.sha256(PREPROCESSING_SOURCE.encode('utf-8')).hexdigest()
(RUN_DIR / 'preprocessing_source.py').write_text(PREPROCESSING_SOURCE, encoding='utf-8')
(RUN_DIR / 'preprocessing.json').write_text(json.dumps(PREPROCESSING, ensure_ascii=False, indent=2), encoding='utf-8')
if USE_SAVED_PREPARATION:
    if PREPROCESSING_SOURCE != snapshot_preprocessing_source or PREPROCESSING != snapshot_preprocessing:
        raise ValueError('The preparation definition differs from the saved snapshot. Create a new preparation version before building/reusing images.')

## 4. Create controlled printed examples

The 12 uploaded font files currently represent ten conservative design groups: Noto Sans Arabic/UI stay together, as do Noto Naskh Arabic/UI. Those paired variants share a split. For this prototype, seven groups train, two validate, and one is held out; the 12 independent-family 8/2/2 target remains for the final study. The notebook creates a 33-character preview sheet for every discovered family/style and writes a manifest with hashes and group assignment.

In [ ]:
# BLOCK 7 — Load frozen font groups
def discover_fonts(font_root):
    found = {}
    if not font_root.exists():
        return found
    for family_dir in sorted(p for p in font_root.iterdir() if p.is_dir()):
        paths = [p for p in family_dir.iterdir() if p.suffix.lower() in {'.ttf', '.otf', '.ttc'}]
        if not paths:
            continue
        regulars = [p for p in paths if not any(tag in p.stem.lower() for tag in ['bold', 'italic', 'black', 'heavy'])]
        bolds = [p for p in paths if 'bold' in p.stem.lower()]
        regular = next((p for p in regulars if p.stem.lower() in {'regular', 'normal'}), None) or (regulars[0] if regulars else paths[0])
        found[family_dir.name] = {'regular': regular, 'bold': bolds[0] if bolds else None}
    return found

# Related variants map to one design group; edit this table before adding new families.
FONT_GROUP_ALIASES = {
    'gandom': 'gandom', 'parastoo': 'parastoo', 'sahel': 'sahel', 'samim': 'samim',
    'shabnam': 'shabnam', 'tanha': 'tanha', 'vazirmatn': 'vazirmatn',
    'vazirmatnregular': 'vazirmatn',
    'notosansarabic': 'noto_sans_arabic', 'notosansarabicui': 'noto_sans_arabic',
    'notonaskharabic': 'noto_naskh_arabic', 'notonaskharabicui': 'noto_naskh_arabic',
    'notokufiarabic': 'noto_kufi_arabic'
}
FONT_GROUP_SPLITS = {
    'gandom': 'train', 'parastoo': 'train', 'sahel': 'train', 'samim': 'train',
    'shabnam': 'train', 'tanha': 'train', 'vazirmatn': 'train',
    'noto_sans_arabic': 'val', 'noto_naskh_arabic': 'val', 'noto_kufi_arabic': 'test'
}

def normalized_font_name(name):
    return re.sub(r'[^a-z0-9]', '', name.casefold())

if USE_SAVED_PREPARATION:
    family_groups = font_splits['family_to_group']
    assert font_splits['group_split'] == FONT_GROUP_SPLITS, 'The saved font groups differ from the frozen assignments.'
    font_split = font_splits['family_split']
    font_families = {}  # Font files are not opened while restoring prepared data.
    (RUN_DIR / 'font_splits.json').write_text(json.dumps(font_splits, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Reused frozen font groups:', len(set(family_groups.values())))
else:
    font_families = discover_fonts(FONT_DIR)
    family_groups = {}
    for family in sorted(font_families):
        normalized = normalized_font_name(family)
        group = FONT_GROUP_ALIASES.get(normalized)
        if group is None:
            raise ValueError(f'Font family {family!r} has no frozen design-group assignment. Add it to FONT_GROUP_ALIASES and FONT_GROUP_SPLITS.')
        family_groups[family] = group
    unknown_groups = sorted(set(family_groups.values()) - set(FONT_GROUP_SPLITS))
    if unknown_groups:
        raise ValueError(f'Font design groups need a frozen split: {unknown_groups}')
    font_split = {family: FONT_GROUP_SPLITS[group] for family, group in family_groups.items()}
    font_splits = {'group_split': FONT_GROUP_SPLITS, 'family_to_group': family_groups,
                   'family_split': font_split, 'target_final_design_groups': 12}
    (RUN_DIR / 'font_splits.json').write_text(json.dumps(font_splits, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Font files/families:', len(font_families), '| independent design groups:', len(set(family_groups.values())))
    print('Group split:', {s: sorted(g for g, assigned in FONT_GROUP_SPLITS.items() if assigned == s) for s in ['train', 'val', 'test']})
    print('Family split:', font_split)
    if font_families and not features.check_feature('raqm'):
        raise RuntimeError('Pillow RAQM Persian shaping support is required to render the printed set.')

In [ ]:
# BLOCK 8 — Load saved printed data and checks (skips font generation in reuse mode)
def build_font_sheet(rows, family, style, output_dir):
    subset = rows[(rows['font_family'] == family) & (rows['font_style'] == style) & (rows['font_size'] == 42)]
    if set(subset['character']) != set(labels['class_order']):
        return None
    cell_w, cell_h, columns = 110, 120, 8
    sheet = Image.new('RGB', (cell_w * columns, cell_h * math.ceil(len(labels['class_order']) / columns)), 'white')
    draw = ImageDraw.Draw(sheet)
    for _, item in subset.iterrows():
        idx = int(item['target_idx'])
        x, y = (idx % columns) * cell_w, (idx // columns) * cell_h
        draw.rectangle((x, y, x + cell_w - 1, y + cell_h - 1), outline='#cccccc')
        with Image.open(item['image_path']) as sample:
            glyph = sample.convert('L').resize((76, 76))
        sheet.paste(glyph.convert('RGB'), (x + 17, y + 4))
        draw.text((x + 5, y + 84), f"{idx}: U+{ord(item['character']):04X}", fill='#333333')
    safe = re.sub(r'[^A-Za-z0-9_-]+', '_', f'{family}_{style}')
    path = output_dir / f'{safe}.png'
    path.parent.mkdir(parents=True, exist_ok=True)
    sheet.save(path)
    return path

def generate_printed_samples():
    if not font_families:
        print('No fonts found; add the supplied Persian-compatible font files to the Drive fonts folder.')
        return pd.DataFrame(), [], []
    if not features.check_feature('raqm'):
        raise RuntimeError('Persian text shaping support (Pillow RAQM) is required for font rendering.')
    output = RUN_DIR / 'printed_generated'
    output.mkdir(parents=True, exist_ok=True)
    font_hashes = {str(path): sha256_file(path) for family in font_families
                   for path in font_families[family].values() if path is not None}
    rows = []
    sizes = [24, 30, 36, 42, 48, 54]
    for family in sorted(font_families):
        for style, font_path in font_families[family].items():
            if font_path is None:
                continue
            for character in labels['class_order']:
                idx = target_index[character]
                for size in sizes:
                    font = ImageFont.truetype(str(font_path), size=size)
                    bbox = ImageDraw.Draw(Image.new('L', (1, 1), 255)).textbbox(
                        (0, 0), character, font=font, direction='rtl', language='fa')
                    glyph_w, glyph_h = bbox[2] - bbox[0], bbox[3] - bbox[1]
                    size_fraction = (size - min(sizes)) / max(1, max(sizes) - min(sizes))
                    padding_fraction = 0.55 - 0.35 * size_fraction
                    padding = max(3, round(max(glyph_w, glyph_h) * padding_fraction))
                    canvas = Image.new('L', (glyph_w + 2 * padding, glyph_h + 2 * padding), 255)
                    glyph_draw = ImageDraw.Draw(canvas)
                    glyph_draw.text((padding - bbox[0], padding - bbox[1]), character, font=font, fill=0,
                                    direction='rtl', language='fa')
                    prepared = prepare_image(canvas)
                    image_path = output / f'class_{idx:02d}' / family / str(size) / f'{style}.png'
                    image_path.parent.mkdir(parents=True, exist_ok=True)
                    prepared.save(image_path)
                    rows.append({'image_path': str(image_path), 'split': font_split[family], 'character': character,
                                 'target_idx': idx, 'source_type': 'printed', 'source_group': family_groups[family],
                                 'font_family': family, 'font_design_group': family_groups[family],
                                 'font_sha256': font_hashes[str(font_path)], 'font_size': size,
                                 'font_style': style, 'is_prepared': True})
    result = pd.DataFrame(rows)
    result.to_csv(RUN_DIR / 'printed_manifest.csv', index=False)
    preview_dir = RUN_DIR / 'font_previews'
    previews = []
    font_render_checks = []
    for family in sorted(font_families):
        for style, font_path in font_families[family].items():
            if font_path is None:
                continue
            preview_path = build_font_sheet(result, family, style, preview_dir)
            if preview_path is not None:
                previews.append(preview_path)
                display(Image.open(preview_path))
            subset = result[(result['font_family'] == family) & (result['font_style'] == style) & (result['font_size'] == 42)]
            by_character = {row['character']: row['image_path'] for _, row in subset.iterrows()}
            blank = []
            for character in labels['class_order']:
                if character not in by_character:
                    blank.append(character)
                    continue
                with Image.open(by_character[character]) as sample:
                    if not np.any(np.asarray(sample.convert('L')) < 245):
                        blank.append(character)
            if 'ا' in by_character and 'آ' in by_character:
                with Image.open(by_character['ا']) as alef_image, Image.open(by_character['آ']) as maddah_image:
                    difference_pixels = int(np.count_nonzero(np.asarray(alef_image.convert('L')) != np.asarray(maddah_image.convert('L'))))
            else:
                difference_pixels = 0
            check = {'font_family': family, 'style': style, 'rendered_class_count': len(by_character),
                     'blank_or_missing_characters': blank, 'alef_maddah_difference_pixels': difference_pixels}
            font_render_checks.append(check)
            if blank or difference_pixels == 0:
                raise ValueError(f'Font render check failed for {family}/{style}: {check}')
    (RUN_DIR / 'font_render_check.json').write_text(json.dumps({'checks': font_render_checks,
        'visual_preview_status': 'generated for every font/style; independent Persian-reader review pending'},
        ensure_ascii=False, indent=2), encoding='utf-8')
    return result, previews, font_render_checks

if USE_SAVED_PREPARATION:
    font_preview_paths = []
    shutil.copy2(PREPARATION_SOURCE_DIR / 'font_render_check.json', RUN_DIR / 'font_render_check.json')
    print('Reused printed manifest and render checks; font images were not regenerated.')
else:
    printed, font_preview_paths, font_render_checks = generate_printed_samples()
    if len(printed):
        print('Printed images:', len(printed))
        print(printed.groupby(['split', 'character']).size().groupby(level=0).agg(['min', 'max']))
        print(f'Generated {len(font_preview_paths)} per-font/style sheets. Each sheet shows all 33 rendered labels.')
        print('Blank/missing glyphs:', sum(len(check['blank_or_missing_characters']) for check in font_render_checks))
        printed_coverage = {split: set(printed.loc[printed['split'] == split, 'character']) for split in ['train', 'val', 'test']}
        print('Missing printed labels by split:', {s: sorted(set(labels['class_order']) - printed_coverage[s]) for s in printed_coverage})

## 5. Optional separately labeled handwritten `آ`

Manual samples are optional for the prototype. If at least three writer folders exist, the notebook records a stable writer-to-split registry before including them. Existing assignments are preserved when new writers are added. Running the notebook never marks the samples reader-reviewed.

In [ ]:
# BLOCK 9 — Load saved manual records
AA_DIR = PROJECT_DIR / 'additional_handwriting' / 'آ'
WRITER_SPLITS_FILE = AA_DIR.parent / 'writer_splits.json'

def load_handwritten_aa():
    if not AA_DIR.exists():
        return pd.DataFrame()
    writers = sorted(p.name for p in AA_DIR.iterdir() if p.is_dir())
    if len(writers) < 3:
        print(f'Handwritten آ not added: need at least 3 writer folders for disjoint train/val/test; found {len(writers)}.')
        return pd.DataFrame()
    if WRITER_SPLITS_FILE.exists():
        registry = json.loads(WRITER_SPLITS_FILE.read_text(encoding='utf-8'))
    else:
        registry = {}
    split_order = ['train', 'val', 'test']
    counts = {split: sum(assigned == split for assigned in registry.values()) for split in split_order}
    for writer in writers:
        if writer in registry:
            continue
        least = min(counts.values())
        choices = [split for split in split_order if counts[split] == least]
        tie = int(hashlib.sha256(f'{SEED}:{writer}'.encode()).hexdigest()[:8], 16) % len(choices)
        chosen = choices[tie]
        registry[writer] = chosen
        counts[chosen] += 1
    if set(split_order) - set(registry[w] for w in writers):
        raise ValueError('The stable writer registry must place at least one current writer in each split.')
    WRITER_SPLITS_FILE.parent.mkdir(parents=True, exist_ok=True)
    WRITER_SPLITS_FILE.write_text(json.dumps(registry, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    rows = []
    for writer in writers:
        for image_path in sorted((AA_DIR / writer).glob('*')):
            if not image_path.is_file() or image_path.suffix.lower() not in {'.png', '.jpg', '.jpeg'}:
                continue
            with Image.open(image_path) as check_image:
                check_image.verify()
            rows.append({'image_path': str(image_path), 'split': registry[writer], 'character': 'آ',
                         'target_idx': target_index['آ'], 'source_type': 'handwritten',
                         'source_group': 'aa-writer:' + writer, 'writer_id': writer, 'is_prepared': False})
    result = pd.DataFrame(rows)
    result.to_csv(RUN_DIR / 'handwritten_aa_manifest.csv', index=False)
    return result

if not USE_SAVED_PREPARATION:
    handwritten_aa = load_handwritten_aa()
else:
    print('Reused manual handwriting records from the preparation snapshot.')

print('Handwritten آ samples found:', len(handwritten_aa))
if len(handwritten_aa):
    display(handwritten_aa.groupby('split').agg(images=('image_path', 'count'), writers=('writer_id', 'nunique')))
    print('Stable writer registry:', WRITER_SPLITS_FILE)

In [ ]:
# BLOCK 10 — Check readiness and save current-run metadata
if USE_SAVED_PREPARATION:
    training_manifest = saved_manifest.copy()
    training_manifest['target_idx'] = training_manifest['target_idx'].astype(int)
    training_manifest.to_csv(RUN_DIR / 'training_manifest.csv', index=False)
else:
    archive_rows = handwriting[['archive_member', 'split', 'character', 'target_idx', 'source_group', 'source_type', 'is_prepared']].copy()
    archive_rows['image_path'] = ''
    archive_rows['writer_id'] = ''
    base_columns = ['archive_member', 'image_path', 'split', 'character', 'target_idx', 'source_group', 'source_type', 'is_prepared', 'writer_id']
    printed_rows = printed[['image_path', 'split', 'character', 'target_idx', 'source_group', 'source_type', 'is_prepared']].copy() if len(printed) else pd.DataFrame(columns=base_columns)
    printed_rows['archive_member'] = ''
    printed_rows['writer_id'] = ''
    aa_rows = handwritten_aa[['image_path', 'split', 'character', 'target_idx', 'source_group', 'source_type', 'is_prepared', 'writer_id']].copy() if len(handwritten_aa) else pd.DataFrame(columns=base_columns)
    aa_rows['archive_member'] = ''
    training_manifest = pd.concat([archive_rows[base_columns], printed_rows[base_columns], aa_rows[base_columns]], ignore_index=True).fillna('')
    training_manifest['target_idx'] = training_manifest['target_idx'].astype(int)
    training_manifest.to_csv(RUN_DIR / 'training_manifest.csv', index=False)

splits = ['train', 'val', 'test']
all_expected = set(labels['class_order'])
archive_expected = set(labels['class_order'][:32])
printed_coverage = {s: set(printed.loc[printed['split'] == s, 'character']) if len(printed) else set() for s in splits}
archive_coverage = {s: set(handwriting.loc[handwriting['split'] == s, 'character']) for s in splits}
all_coverage = {s: set(training_manifest.loc[training_manifest['split'] == s, 'character']) for s in splits}
missing_printed = {s: sorted(all_expected - printed_coverage[s]) for s in splits}
missing_archive = {s: sorted(archive_expected - archive_coverage[s]) for s in splits}
missing_any = {s: sorted(all_expected - all_coverage[s]) for s in splits}
FONT_RENDERING_READY = bool(font_render_checks) and all(not check['blank_or_missing_characters'] and check['alef_maddah_difference_pixels'] > 0 for check in font_render_checks)
PROTOTYPE_READY = (FONT_RENDERING_READY and not missing_printed['train'] and not missing_printed['val']
                   and not missing_archive['train'] and not missing_archive['val'])
handwritten_aa_by_split = {s: int(((training_manifest['split'] == s) & (training_manifest['character'] == 'آ')
                                    & (training_manifest['source_type'] == 'handwritten')).sum()) for s in splits}
FINAL_READY = (LABEL_REVIEW_APPROVED and bool(LABEL_REVIEWER.strip()) and PROTOTYPE_READY
               and all(not missing_printed[s] for s in splits)
               and all(handwritten_aa_by_split[s] > 0 for s in splits)
               and len(set(family_groups.values())) >= 12)

manifest_hashes = {name: sha256_file(RUN_DIR / name) for name in ['training_manifest.csv', 'printed_manifest.csv'] if (RUN_DIR / name).exists()}
metadata = {'run_id': RUN_ID, 'mode': EXPERIMENT_MODE, 'preparation_source': str(PREPARATION_SOURCE_DIR if USE_SAVED_PREPARATION else RUN_DIR), 'archive_sha256': EXPECTED_ARCHIVE_SHA256,
            'label_version': labels['label_version'], 'class_order': labels['class_order'], 'seed': SEED,
            'model': 'DenseNet121 ImageNet transfer learning', 'preprocessing': PREPROCESSING,
            'preprocessing_source_sha256': PREPROCESSING['source_sha256'],
            'font_groups': font_splits, 'manifest_sha256': manifest_hashes,
            'manifest_counts': {s: int((training_manifest['split'] == s).sum()) for s in splits},
            'font_render_check_sha256': sha256_file(RUN_DIR / 'font_render_check.json') if (RUN_DIR / 'font_render_check.json').exists() else None,
            'font_rendering_ready': bool(FONT_RENDERING_READY),
            'coverage': {'printed_missing': missing_printed, 'archive_handwriting_missing': missing_archive,
                         'any_source_missing': missing_any, 'handwritten_aa_samples': handwritten_aa_by_split},
            'reader_review': 'approved' if LABEL_REVIEW_APPROVED and LABEL_REVIEWER.strip() else 'pending; prototype labels provisional',
            'prototype_ready': bool(PROTOTYPE_READY), 'final_ready': bool(FINAL_READY), 'final_test_used': False}
(RUN_DIR / 'run_config.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print('Missing printed labels by split:', missing_printed)
print('Missing archive handwritten labels by split:', missing_archive)
print('Handwritten آ images by split:', handwritten_aa_by_split)
print('Prototype preparation ready:', PROTOTYPE_READY)
print('Final-study gate ready:', FINAL_READY)
print('Versioned manifest and config:', RUN_DIR)

if EXPERIMENT_MODE == 'prepare':
    print('Phase 2 data preparation complete. Final test data is reserved. Set EXPERIMENT_MODE="prototype" for the separate Phase 3 training run.')
assert training_manifest.groupby(['source_type', 'source_group'])['split'].nunique().max() == 1, 'A source group crosses partitions.'
assert printed.apply(lambda row: font_splits['group_split'][row['source_group']] == row['split'], axis=1).all(), 'A printed design group changed split.'
assert training_manifest.apply(lambda row: labels['class_order'][int(row['target_idx'])] == row['character'], axis=1).all()
if not USE_SAVED_PREPARATION:
    SOURCE_MANIFEST_SHA256 = sha256_file(RUN_DIR / 'training_manifest.csv')
    source_config = json.loads(json.dumps(metadata))

## 6. Save/restore prepared images once

The cache is bound to the immutable preparation snapshot, selected sample IDs, label order, frozen groups, and exact preparation code/settings. An incomplete cache is rebuilt; a corrupt completed cache is rejected. Changing raw data requires a new preparation snapshot. The first build prepares the selected train/validation images and writes one ZIP to Drive. Later runs restore it to local storage without opening raw image/font inputs.

In [ ]:
# BLOCK 11 — BUILD ONCE or LOAD prepared images from Google Drive
"""Persistent, deterministic prepared-image snapshots for the training notebook.

Callers must supply an immutable preparation manifest and its verified hash. Editing
source images or adding samples requires a new preparation snapshot. Cache hits read
only the cache package and metadata, never the original archive, fonts, or images.
Random augmentation and model normalization belong in the training loader.
"""

import hashlib
import io
import json
import os
import re
import shutil
import stat
import tempfile
import zipfile
from pathlib import Path, PurePosixPath

import pandas as pd
from PIL import Image, ImageOps


CACHE_SCHEMA = 2
METADATA_FILES = (
    "prepared_manifest.csv", "prepared_images.zip", "preprocessing.json",
    "preprocessing_source.py", "font_splits.json", "source_run_config.json",
)


class PreparedCacheError(RuntimeError):
    """A completed cache is corrupt or incompatible with its recorded identity."""


def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _json_bytes(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False).encode("utf-8")


def _plain(value):
    if isinstance(value, dict):
        return {str(key): _plain(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_plain(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if value is None or (not isinstance(value, (str, bool)) and pd.isna(value)):
        return ""
    if hasattr(value, "item"):
        return value.item()
    return value


def _stable_config(value):
    """Exclude execution locations/times; retain source provenance and review state."""
    if isinstance(value, dict):
        return {str(key): _stable_config(item) for key, item in value.items()
                if str(key).lower() not in {"run_id", "run_dir", "runtime", "runtime_dir",
                                            "runtime_root", "created_at", "generated_at"}
                and not str(key).lower().startswith("runtime_")}
    if isinstance(value, (list, tuple)):
        return [_stable_config(item) for item in value]
    return _plain(value)


def _as_bool(value):
    return value if isinstance(value, bool) else str(value).strip().lower() in {"true", "1", "yes"}


def select_prototype_manifest(frame, seed=42, label_review_approved=False):
    """Keep the existing 100/25 archive sampling rule within frozen train/val splits."""
    chosen = []
    for split, cap in (("train", 100), ("val", 25)):
        part = frame[frame["split"] == split]
        archive = part[(part["source_type"] == "handwritten") & (part["character"] != "آ")]
        for _, group in archive.groupby("character", sort=True):
            chosen.append(group.sample(n=min(cap, len(group)), random_state=seed))
        printed = part[part["source_type"] == "printed"]
        if len(printed):
            chosen.append(printed)
        if label_review_approved:
            manual = part[(part["source_type"] == "handwritten") & (part["character"] == "آ")]
            if len(manual):
                chosen.append(manual.sample(n=min(cap, len(manual)), random_state=seed))
    if not chosen:
        return frame.iloc[:0].copy().reset_index(drop=True)
    return pd.concat(chosen, ignore_index=True).drop_duplicates(
        subset=["split", "source_group", "archive_member", "image_path"]
    ).reset_index(drop=True)


def _validate_frame(frame, allow_test):
    required = {"archive_member", "image_path", "source_group", "source_type",
                "split", "character", "target_idx"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"Prepared-image manifest is missing columns: {sorted(missing)}")
    if frame.empty:
        raise ValueError("Cannot cache an empty prepared-image manifest.")
    allowed = {"train", "val", "test"} if allow_test else {"train", "val"}
    invalid = set(frame["split"]) - allowed
    if invalid:
        raise ValueError(f"Refusing image reads from disallowed split(s): {sorted(invalid)}")
    if (frame.groupby(["source_type", "source_group"])["split"].nunique() > 1).any():
        raise ValueError("Source groups overlap between splits.")
    source_ids = frame.apply(lambda row: "archive:" + row["archive_member"] if row["archive_member"]
                            else "image:" + row["image_path"], axis=1)
    if (pd.DataFrame({"source": source_ids, "split": frame["split"]})
            .groupby("source")["split"].nunique() > 1).any():
        raise ValueError("Source images overlap between splits.")
    targets = frame["target_idx"].astype(int)
    if not targets.between(0, 32).all():
        raise ValueError("Prepared data must use the stable 33-label target range.")


def _safe_relative(value, prefix=None):
    path = PurePosixPath(str(value))
    if (not str(value) or "\\" in str(value) or path.is_absolute()
            or any(part in {".", "..", ""} for part in path.parts)
            or path.parts[0] not in {"images", "previews"} or len(path.parts) != 2
            or (prefix is not None and path.parts[0] != prefix)):
        raise PreparedCacheError(f"Unsafe prepared-image path: {value!r}")
    return path


def _read_json(path):
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except (OSError, ValueError) as exc:
        raise PreparedCacheError(f"Cannot read cache metadata: {path}") from exc


def _verify_completed(cache_dir, identity, cache_id):
    config = _read_json(cache_dir / "cache_config.json")
    if (config.get("status") != "complete" or config.get("cache_schema") != CACHE_SCHEMA
            or config.get("cache_id") != cache_id or config.get("identity") != identity):
        raise PreparedCacheError("Completed cache has an incompatible identity or status.")
    hashes = config.get("files_sha256", {})
    if set(hashes) != set(METADATA_FILES):
        raise PreparedCacheError("Completed cache is missing required metadata hashes.")
    for name in METADATA_FILES:
        path = cache_dir / name
        if not path.is_file() or _sha256(path) != hashes[name]:
            raise PreparedCacheError(f"Completed cache failed its SHA-256 check: {name}")
    try:
        prepared = pd.read_csv(cache_dir / "prepared_manifest.csv", keep_default_na=False)
    except (OSError, ValueError) as exc:
        raise PreparedCacheError("Cannot read prepared-image manifest.") from exc
    if len(prepared) != config.get("image_count"):
        raise PreparedCacheError("Prepared-image count differs from the completed metadata.")
    for name in ["prepared_relative_path", "prepared_sha256", "source_sha256"]:
        if name not in prepared or (prepared[name].astype(str).str.len() == 0).any():
            raise PreparedCacheError(f"Prepared manifest lacks {name} values.")
    if not prepared["prepared_relative_path"].is_unique:
        raise PreparedCacheError("Prepared-image manifest contains duplicate output paths.")
    for relative in prepared["prepared_relative_path"]:
        _safe_relative(relative, "images")
    for name in ["raw_preview_relative_path", "raw_preview_sha256"]:
        if name not in prepared:
            raise PreparedCacheError(f"Prepared manifest lacks {name}.")
    preview_mask = prepared["raw_preview_relative_path"].astype(str).ne("")
    if not preview_mask.equals(prepared["split"].eq("val")):
        raise PreparedCacheError("Raw preview thumbnails must cover validation rows only.")
    if (prepared.loc[preview_mask, "raw_preview_sha256"].astype(str).str.len() != 64).any():
        raise PreparedCacheError("Validation thumbnails lack their SHA-256 values.")
    if not prepared.loc[preview_mask, "raw_preview_relative_path"].is_unique:
        raise PreparedCacheError("Validation thumbnail paths are duplicated.")
    for relative in prepared.loc[preview_mask, "raw_preview_relative_path"]:
        _safe_relative(relative, "previews")
    return prepared, config


def _has_completed_marker(cache_dir):
    marker = cache_dir / "cache_config.json"
    if not marker.is_file():
        return False
    status = _read_json(marker).get("status")
    if status == "complete":
        return True
    if status in {"building", "incomplete"}:
        return False
    raise PreparedCacheError("Cache metadata has an unrecognized completion status.")


def _runtime_matches(folder, prepared, output_size):
    if not folder.is_dir():
        return False
    for row in prepared.itertuples(index=False):
        path = folder / row.prepared_relative_path
        if not path.is_file() or _sha256(path) != row.prepared_sha256:
            return False
        try:
            with Image.open(path) as image:
                if image.mode != "L" or image.size != (output_size, output_size):
                    return False
        except OSError:
            return False
        if row.raw_preview_relative_path:
            preview_path = folder / row.raw_preview_relative_path
            if not preview_path.is_file() or _sha256(preview_path) != row.raw_preview_sha256:
                return False
            try:
                with Image.open(preview_path) as preview:
                    if (preview.mode != "RGB" or min(preview.size) <= 0
                            or max(preview.size) > 112):
                        return False
            except OSError:
                return False
    return True


def _restore(cache_dir, runtime_root, cache_id, prepared, output_size):
    runtime_root.mkdir(parents=True, exist_ok=True)
    destination = runtime_root / cache_id
    if not _runtime_matches(destination, prepared, output_size):
        staging = Path(tempfile.mkdtemp(prefix=f".{cache_id}.restore-", dir=runtime_root))
        try:
            expected = set(prepared["prepared_relative_path"])
            expected.update(relative for relative in prepared["raw_preview_relative_path"] if relative)
            # A Drive-mounted ZIP is slow to seek for each member. Copy it in one
            # sequential stream, then perform member reads on runtime storage.
            local_package = staging / "prepared_images.zip"
            with (cache_dir / "prepared_images.zip").open("rb") as source, local_package.open("wb") as out:
                shutil.copyfileobj(source, out, length=1024 * 1024)
            with zipfile.ZipFile(local_package) as archive:
                infos = archive.infolist()
                if len(infos) != len(expected) or {entry.filename for entry in infos} != expected:
                    raise PreparedCacheError("Prepared ZIP members differ from its manifest.")
                for entry in infos:
                    _safe_relative(entry.filename)
                    if entry.is_dir() or stat.S_ISLNK(entry.external_attr >> 16):
                        raise PreparedCacheError("Prepared ZIP must contain ordinary PNG files only.")
                    target = staging / entry.filename
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with archive.open(entry) as source, target.open("wb") as out:
                        shutil.copyfileobj(source, out)
            local_package.unlink()
            if not _runtime_matches(staging, prepared, output_size):
                raise PreparedCacheError("Extracted prepared images failed integrity or shape checks.")
            if destination.exists():
                shutil.rmtree(destination)
            os.replace(staging, destination)
        except (OSError, zipfile.BadZipFile) as exc:
            raise PreparedCacheError("Cannot restore the prepared-image package.") from exc
        finally:
            if staging.exists():
                shutil.rmtree(staging)
    result = prepared.copy()
    result["prepared_image_path"] = [str(destination / relative)
                                     for relative in result["prepared_relative_path"]]
    result["raw_preview_image_path"] = [str(destination / relative) if relative else ""
                                        for relative in result["raw_preview_relative_path"]]
    result["is_prepared"] = True
    return result


def ensure_prepared_cache(frame, cache_root, runtime_root, preprocessing, preprocessing_source,
                          labels_sha256, source_manifest_sha256, archive_sha256, font_splits,
                          source_config, prepare_fn, ensure_archive, scope="prototype",
                          allow_test=False):
    """Build once or restore a verified package of deterministic prepared PNGs.

    ``ensure_archive()`` is called lazily on a miss only if selected archive rows
    exist, and returns the verified original ZIP path. The caller verifies the
    immutable input manifest against ``source_manifest_sha256`` before this call.
    A completed corrupt cache raises PreparedCacheError; incomplete builds may be
    rebuilt. Returned paths point to runtime storage, not individual Drive files.
    """
    if not re.fullmatch(r"[A-Za-z0-9_-]+", scope):
        raise ValueError("Cache scope must be a simple name, such as prototype or final.")
    frame = frame.copy().astype(object).where(frame.notna(), "").reset_index(drop=True)
    _validate_frame(frame, allow_test)
    output_size = int(preprocessing.get("output_size", 224))
    if output_size != 224:
        raise ValueError("DenseNet121 prepared inputs must remain 224 by 224.")
    source_sha = hashlib.sha256(preprocessing_source.encode("utf-8")).hexdigest()
    if preprocessing.get("source_sha256", source_sha) != source_sha:
        raise ValueError("Preparation source differs from its declared SHA-256.")
    identity_frame = frame.drop(columns=["prepared_image_path", "prepared_relative_path", "prepared_sha256",
                                         "raw_preview_image_path", "raw_preview_relative_path",
                                         "raw_preview_sha256"], errors="ignore")
    identity = _plain({
        "cache_schema": CACHE_SCHEMA, "scope": scope, "allow_test": bool(allow_test),
        "archive_sha256": archive_sha256, "labels_sha256": labels_sha256,
        "source_manifest_sha256": source_manifest_sha256,
        "preprocessing": preprocessing, "preprocessing_source_sha256": source_sha,
        "font_splits": font_splits, "source_config": _stable_config(source_config),
        "selected_records": identity_frame.to_dict(orient="records"),
    })
    cache_id = f"{scope}_" + hashlib.sha256(_json_bytes(identity)).hexdigest()[:24]
    cache_root, runtime_root = Path(cache_root), Path(runtime_root)
    cache_root.mkdir(parents=True, exist_ok=True)
    runtime_root.mkdir(parents=True, exist_ok=True)
    cache_dir = cache_root / cache_id
    marker = cache_dir / "cache_config.json"
    cache_hit = _has_completed_marker(cache_dir)
    if cache_hit:
        prepared, config = _verify_completed(cache_dir, identity, cache_id)
    else:
        staging = Path(tempfile.mkdtemp(prefix=f".{cache_id}.build-", dir=runtime_root))
        durable_staging = None
        raw_archive = None
        try:
            rows = []
            for index, row in frame.iterrows():
                if row["archive_member"]:
                    if raw_archive is None:
                        raw_archive = zipfile.ZipFile(ensure_archive())
                    source_bytes = raw_archive.read(row["archive_member"])
                else:
                    source_bytes = Path(row["image_path"]).read_bytes()
                row_source_sha = hashlib.sha256(source_bytes).hexdigest()
                if row.get("source_sha256") and row["source_sha256"] != row_source_sha:
                    raise ValueError("Source image differs from the saved preparation snapshot.")
                with Image.open(io.BytesIO(source_bytes)) as image:
                    preview_relative = preview_sha = ""
                    if row["split"] == "val":
                        thumbnail = ImageOps.contain(image.convert("RGB"), (112, 112))
                        preview_relative = f"previews/{index:07d}_{row_source_sha[:16]}.png"
                        preview_path = staging / preview_relative
                        preview_path.parent.mkdir(parents=True, exist_ok=True)
                        thumbnail.save(preview_path, format="PNG")
                        preview_sha = _sha256(preview_path)
                    prepared_image = (image.convert("L") if _as_bool(row.get("is_prepared", False))
                                      else prepare_fn(image).convert("L"))
                    if prepared_image.size != (output_size, output_size):
                        raise ValueError(f"Prepared image has unexpected shape: {prepared_image.size}")
                    relative = f"images/{index:07d}_{row_source_sha[:16]}.png"
                    output = staging / relative
                    output.parent.mkdir(parents=True, exist_ok=True)
                    prepared_image.save(output, format="PNG")
                record = row.to_dict()
                record.update(prepared_relative_path=relative, prepared_sha256=_sha256(output),
                              source_sha256=row_source_sha, is_prepared=True,
                              raw_preview_relative_path=preview_relative, raw_preview_sha256=preview_sha)
                record.pop("prepared_image_path", None)
                record.pop("raw_preview_image_path", None)
                rows.append(record)
            prepared = pd.DataFrame(rows)
            prepared.to_csv(staging / "prepared_manifest.csv", index=False)
            # PNG already compresses its pixels; storing avoids a second pass.
            with zipfile.ZipFile(staging / "prepared_images.zip", "w", compression=zipfile.ZIP_STORED) as archive:
                for relative in prepared["prepared_relative_path"]:
                    archive.write(staging / relative, relative)
                for relative in prepared["raw_preview_relative_path"]:
                    if relative:
                        archive.write(staging / relative, relative)
            payloads = {"preprocessing.json": preprocessing, "font_splits.json": font_splits,
                        "source_run_config.json": source_config}
            for name, payload in payloads.items():
                (staging / name).write_text(json.dumps(_plain(payload), ensure_ascii=False, indent=2) + "\n",
                                           encoding="utf-8")
            (staging / "preprocessing_source.py").write_text(preprocessing_source, encoding="utf-8")
            config = {"cache_schema": CACHE_SCHEMA, "cache_id": cache_id, "status": "complete",
                      "identity": identity, "image_count": len(prepared),
                      "split_counts": prepared["split"].value_counts().to_dict(),
                      "files_sha256": {name: _sha256(staging / name) for name in METADATA_FILES}}
            # Publish complete metadata last. Partially copied packages never count as hits.
            durable_staging = Path(tempfile.mkdtemp(prefix=f".{cache_id}.publish-", dir=cache_root))
            for name in METADATA_FILES:
                shutil.copy2(staging / name, durable_staging / name)
            marker_temp = durable_staging / "cache_config.json.tmp"
            marker_temp.write_text(json.dumps(_plain(config), ensure_ascii=False, indent=2) + "\n",
                                   encoding="utf-8")
            os.replace(marker_temp, durable_staging / "cache_config.json")
            if cache_dir.exists():
                if _has_completed_marker(cache_dir):
                    prepared, config = _verify_completed(cache_dir, identity, cache_id)
                else:
                    shutil.rmtree(cache_dir)
                    os.replace(durable_staging, cache_dir)
            else:
                os.replace(durable_staging, cache_dir)
            prepared, config = _verify_completed(cache_dir, identity, cache_id)
        finally:
            if raw_archive is not None:
                raw_archive.close()
            if staging.exists():
                shutil.rmtree(staging)
            if durable_staging is not None and durable_staging.exists():
                shutil.rmtree(durable_staging)
    prepared = _restore(cache_dir, runtime_root, cache_id, prepared, output_size)
    return prepared, {"cache_id": cache_id, "cache_dir": str(cache_dir), "cache_hit": cache_hit,
                      "metadata": config}


# Cache only the selected prototype train/validation rows; no final-test image is opened.
PREPARED_DATA_READY = False
if PROTOTYPE_READY and EXPERIMENT_MODE in {'prepare', 'prototype'}:
    selected = select_prototype_manifest(training_manifest, seed=SEED, label_review_approved=LABEL_REVIEW_APPROVED)
    cached_prototype_manifest, cache_info = ensure_prepared_cache(
        selected, CACHE_ROOT, RUNTIME_CACHE_ROOT, PREPROCESSING, PREPROCESSING_SOURCE,
        sha256_file(CHARACTER_MAP), SOURCE_MANIFEST_SHA256, EXPECTED_ARCHIVE_SHA256,
        font_splits, source_config, prepare_image, ensure_local_archive,
        scope=f'prototype_seed_{SEED}', allow_test=False)
    cached_prototype_manifest.to_csv(RUN_DIR / 'prototype_manifest.csv', index=False)
    (RUN_DIR / 'prepared_cache_config.json').write_text(json.dumps(cache_info['metadata'], ensure_ascii=False, indent=2), encoding='utf-8')
    metadata['prepared_dataset'] = {'cache_id': cache_info['cache_id'], 'cache_dir': cache_info['cache_dir'],
        'scope': f'prototype_seed_{SEED}', 'samples': len(cached_prototype_manifest), 'final_test_used': False}
    metadata['training'] = {'batch_size': 16, 'classifier_epochs': 2, 'fine_tune_epochs': 3,
        'selection': 'equal mean of printed and archive-handwriting validation macro F1',
        'sampling': 'inverse source/character stratum count, replacement=True'}
    metadata['manifest_sha256']['prototype_manifest.csv'] = sha256_file(RUN_DIR / 'prototype_manifest.csv')
    (RUN_DIR / 'run_config.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
    PREPARED_DATA_READY = True
    PREPARED_RUN_ID = RUN_ID
    print('Prepared dataset:', 'REUSED' if cache_info['cache_hit'] else 'BUILT', '|', len(cached_prototype_manifest), 'images')
    print('Durable cache:', cache_info['cache_dir'])
    print('Training reads local PNGs; archive audit, font generation, and deterministic preprocessing are skipped on reuse.')
else:
    print('Prototype cache deferred: inspect readiness, or use the explicitly enabled final-study branch.')


## 7. Prototype and final training

The prototype branch uses seed 42, deterministic subsets from the frozen train/validation assignments (up to 100 archive training and 25 archive validation images per class), two classifier epochs, then up to three final-block fine-tuning epochs. It never opens the final test split. Validation scores are calculated separately for printed 33-class data and archive handwriting 32-class data, then averaged equally. Final training remains behind human-review, font, manual `آ`, and test gates.

In [ ]:
# BLOCK 12 — Load DenseNet121 training functions
if EXPERIMENT_MODE == 'prepare':
    print('Training libraries deferred; Phase 2 is a data-preparation run.')
else:
    try:
        import torch, torchvision
        from torchvision import models, transforms
        from torchvision.transforms import RandomAffine, GaussianBlur
        from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
        from sklearn.metrics import accuracy_score, f1_score, classification_report
    except ImportError as exc:
        raise RuntimeError('Use a Colab runtime with PyTorch, Torchvision, and scikit-learn for training.') from exc

    print('PyTorch:', torch.__version__, '| Torchvision:', torchvision.__version__)
    print('GPU available:', torch.cuda.is_available())
    if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

    MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    TRAIN_AUGMENT = transforms.Compose([
        RandomAffine(degrees=7, translate=(0.05, 0.05), fill=1.0),
        GaussianBlur(kernel_size=3, sigma=(0.1, 0.6))
    ])

    def as_bool(value):
        return value if isinstance(value, bool) else str(value).strip().lower() in {'true', '1', 'yes'}

    class CharacterDataset(Dataset):
        def __init__(self, frame, archive_path=None, training=False):
            self.frame = frame.reset_index(drop=True)
            self.training = training
            if 'prepared_image_path' not in self.frame or not self.frame['prepared_image_path'].astype(str).ne('').all():
                raise ValueError('Restore/build the prepared dataset before constructing training loaders.')
        def __len__(self): return len(self.frame)
        def close(self): pass
        def __getitem__(self, index):
            row = self.frame.iloc[index]
            with Image.open(row['prepared_image_path']) as image:
                if image.mode != 'L' or image.size != (OUTPUT_SIZE, OUTPUT_SIZE):
                    raise ValueError(f"Invalid cached image: {row['prepared_image_path']}")
                array = np.asarray(image, dtype=np.float32).copy() / 255.0
            tensor = torch.from_numpy(array).unsqueeze(0).repeat(3, 1, 1)
            if self.training: tensor = TRAIN_AUGMENT(tensor)
            tensor = (tensor - MEAN) / STD
            return tensor, int(row['target_idx'])

    def build_loaders(frame, batch_size=16, include_test=False):
        active_splits = ['train', 'val'] + (['test'] if include_test else [])
        subsets = {s: frame[frame['split'] == s].reset_index(drop=True) for s in active_splits}
        if any(len(v) == 0 for v in subsets.values()):
            raise ValueError(f'Empty required split(s): {[s for s, v in subsets.items() if len(v) == 0]}')
        train_frame = subsets['train']
        stratum_counts = train_frame.groupby(['source_type', 'character']).size().to_dict()
        weights = [1.0 / stratum_counts[(row.source_type, row.character)] for row in train_frame.itertuples()]
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
        datasets, loaders = {}, {}
        datasets['train'] = CharacterDataset(train_frame, LOCAL_ARCHIVE, training=True)
        loaders['train'] = DataLoader(datasets['train'], batch_size=batch_size, sampler=sampler,
                                      num_workers=0, pin_memory=torch.cuda.is_available())
        for split in active_splits[1:]:
            datasets[split] = CharacterDataset(subsets[split], LOCAL_ARCHIVE, training=False)
            loaders[split] = DataLoader(datasets[split], batch_size=32, shuffle=False, num_workers=0)
        val_by_source = {}
        for source in sorted(subsets['val']['source_type'].unique()):
            source_frame = subsets['val'][subsets['val']['source_type'] == source].reset_index(drop=True)
            source_ds = CharacterDataset(source_frame, LOCAL_ARCHIVE, training=False)
            val_by_source[source] = (DataLoader(source_ds, batch_size=32, shuffle=False, num_workers=0), source_frame, source_ds)
        loaders['val_by_source'] = val_by_source
        return loaders, subsets, datasets

    def make_model(device):
        model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        for parameter in model.parameters(): parameter.requires_grad = False
        model.classifier = torch.nn.Linear(model.classifier.in_features, len(labels['class_order']))
        return model.to(device)

    def predict_loader(model, loader, device):
        model.eval(); ys = []; ps = []; scores = []
        with torch.inference_mode():
            for x, y in loader:
                logits = model(x.to(device))
                probabilities = torch.softmax(logits, dim=1)
                score, prediction = probabilities.max(1)
                ps.extend(prediction.cpu().tolist()); scores.extend(score.cpu().tolist()); ys.extend(y.tolist())
        return np.asarray(ys), np.asarray(ps), np.asarray(scores)

    def evaluate_sources(model, val_by_source, device):
        results = {}
        for source, (loader, frame, _) in val_by_source.items():
            y, p, _ = predict_loader(model, loader, device)
            supported = sorted(set(y.tolist()))
            results[source] = {'samples': int(len(y)), 'supported_classes': len(supported),
                               'macro_f1': float(f1_score(y, p, average='macro', labels=supported, zero_division=0)),
                               'accuracy': float(accuracy_score(y, p))}
        selection_sources = [results[s]['macro_f1'] for s in ('printed', 'handwritten') if s in results]
        if len(selection_sources) != 2:
            raise ValueError('Prototype selection requires printed and archive-handwriting validation sources.')
        return results, float(np.mean(selection_sources))

    def fit_one_seed(seed, loaders, device, prototype=True):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        model = make_model(device)
        best_score = -1.0
        best_path = RUN_DIR / f'densenet121_seed_{seed}_best.pth'
        phases = [('classifier', 2, 1e-3), ('fine_tune', 3, 1e-4)] if prototype else [('classifier', 5, 1e-3), ('fine_tune', 30, 1e-4)]
        for phase, max_epochs, lr in phases:
            if phase == 'fine_tune':
                for parameter in model.features.denseblock4.parameters(): parameter.requires_grad = True
                for parameter in model.features.norm5.parameters(): parameter.requires_grad = True
            optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
            best_phase, stale = -1.0, 0
            for epoch in range(1, max_epochs + 1):
                model.train(); total_loss = 0.0; total = 0
                for x, y in loaders['train']:
                    x, y = x.to(device), y.to(device)
                    optimizer.zero_grad(set_to_none=True)
                    logits = model(x); loss = torch.nn.functional.cross_entropy(logits, y)
                    loss.backward(); optimizer.step()
                    total_loss += float(loss.item()) * len(y); total += len(y)
                source_scores, selection_score = evaluate_sources(model, loaders['val_by_source'], device)
                print(f'seed={seed} {phase} epoch={epoch} loss={total_loss/max(total, 1):.4f} selection={selection_score:.4f} sources={source_scores}')
                if selection_score > best_score:
                    best_score = selection_score
                    torch.save(model.state_dict(), best_path)
                if selection_score > best_phase:
                    best_phase = selection_score; stale = 0
                else:
                    stale += 1
                    if not prototype and phase == 'fine_tune' and stale >= 5: break
            model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))
        return model, best_path, best_score

    def _roman_display(character):
        item = next(row for row in labels['classes'] if row['character'] == character)
        values = item.get('roman_values') or []
        if isinstance(values, str): values = [values]
        return '/'.join(str(value) for value in values) or str(item.get('display', ''))

    def _read_raw_preview_image(row):
        # Input thumbnail was saved during the one-time cache build.
        with Image.open(row['raw_preview_image_path']) as image:
            return image.convert('RGB')

    def _read_preview_image(row):
        with Image.open(row['prepared_image_path']) as image:
            return image.convert('L')

    def export_validation_predictions(model, val_by_source, device):
        rows = []
        previews = {}
        for source, (loader, frame, _) in val_by_source.items():
            y, p, scores = predict_loader(model, loader, device)
            source_frame = frame.reset_index(drop=True)
            if len(source_frame) != len(y):
                raise ValueError(f'Prediction/frame length mismatch for {source}.')
            for i, row in source_frame.iterrows():
                true_character = labels['class_order'][int(y[i])]
                predicted_character = labels['class_order'][int(p[i])]
                rows.append({'source_type': source, 'split': 'val', 'target_idx': int(y[i]),
                    'true_character': true_character, 'true_romanization': _roman_display(true_character),
                    'predicted_idx': int(p[i]), 'predicted_character': predicted_character,
                    'predicted_romanization': _roman_display(predicted_character),
                    'correct': bool(y[i] == p[i]), 'top1_model_score': float(scores[i]),
                    'archive_member': row.get('archive_member', ''), 'image_path': row.get('image_path', '')})
            prediction_frame = pd.DataFrame(rows)
            prediction_frame = prediction_frame[prediction_frame['source_type'] == source].reset_index(drop=True)
            supported = sorted(set(y.tolist()))
            report = classification_report(y, p, labels=supported,
                target_names=[labels['class_order'][i] for i in supported], output_dict=True, zero_division=0)
            pd.DataFrame(report).T.to_csv(RUN_DIR / f'validation_classification_{source}.csv', encoding='utf-8')
            # Pin the two planned demonstrations, then show errors and correct examples.
            special_character = 'ش' if source == 'handwritten' else ('آ' if source == 'printed' else None)
            special = prediction_frame[prediction_frame['true_character'] == special_character].head(1) if special_character else prediction_frame.head(0)
            remaining = prediction_frame.drop(index=special.index)
            errors = remaining[~remaining['correct']].head(4)
            correct = remaining[remaining['correct']].head(max(0, 8 - len(special) - len(errors)))
            selected = pd.concat([special, errors, correct], ignore_index=True).drop_duplicates(
                subset=['archive_member', 'image_path']).head(8)
            previews[source] = (source_frame, y, p, scores, selected)
        result = pd.DataFrame(rows)
        result.to_csv(RUN_DIR / 'validation_predictions.csv', index=False, encoding='utf-8')
        for source, (source_frame, y, p, scores, selected) in previews.items():
            cell_w, cell_h, columns = 300, 310, 3
            count = max(1, len(selected))
            sheet = Image.new('RGB', (cell_w * columns, cell_h * math.ceil(count / columns)), 'white')
            draw = ImageDraw.Draw(sheet)
            for tile, pred_row in selected.iterrows():
                # Map back to the original validation row by its stable member/path pair.
                original_index = next(i for i, row in source_frame.iterrows()
                    if row.get('archive_member', '') == pred_row['archive_member']
                    and row.get('image_path', '') == pred_row['image_path'])
                source_row = source_frame.iloc[original_index]
                raw_image = ImageOps.contain(_read_raw_preview_image(source_row), (112, 112))
                prepared_image = _read_preview_image(source_row).convert('RGB').resize((112, 112))
                x, y0 = (tile % columns) * cell_w, (tile // columns) * cell_h
                raw_tile = Image.new('RGB', (112, 112), 'white')
                raw_tile.paste(raw_image, ((112 - raw_image.width) // 2, (112 - raw_image.height) // 2))
                sheet.paste(raw_tile, (x + 18, y0 + 8))
                sheet.paste(prepared_image, (x + 160, y0 + 8))
                draw.text((x + 52, y0 + 124), 'Input', fill='black')
                draw.text((x + 188, y0 + 124), 'Prepared', fill='black')
                true_character, predicted_character = pred_row['true_character'], pred_row['predicted_character']
                correct = bool(pred_row['correct'])
                color = '#147d32' if correct else '#b42318'
                draw.text((x + 8, y0 + 245), f"True: {_roman_display(true_character)} (U+{ord(true_character):04X})", fill='black')
                draw.text((x + 8, y0 + 265), f"Pred: {_roman_display(predicted_character)} (U+{ord(predicted_character):04X})", fill=color)
                draw.text((x + 8, y0 + 285), f"Top-1 model score: {float(pred_row['top1_model_score']):.3f}", fill='black')
            preview_path = RUN_DIR / f'validation_preview_{source}.png'
            sheet.save(preview_path)
            previews[source] = preview_path
            display(sheet)
            print(f'{source} validation preview:', preview_path)
        return result, previews

    def export_prototype_bundle(model, checkpoint_path, source_metrics, prediction_frame):
        bundle_dir = RUN_DIR / 'prototype_model_bundle'
        bundle_dir.mkdir(parents=True, exist_ok=True)
        torch.save(model.state_dict(), bundle_dir / 'model_state_dict.pth')
        label_payload = {'label_version': labels['label_version'], 'class_order': labels['class_order'],
                         'classes': labels['classes'], 'romanization_note': 'Roman display values may have context-dependent alternatives.'}
        (bundle_dir / 'labels.json').write_text(json.dumps(label_payload, ensure_ascii=False, indent=2), encoding='utf-8')
        model_spec = {'architecture': 'torchvision DenseNet121', 'weights': 'ImageNet pretrained initialization',
                      'classifier_outputs': len(labels['class_order']), 'input_shape': [3, OUTPUT_SIZE, OUTPUT_SIZE],
                      'normalization_mean': [0.485, 0.456, 0.406], 'normalization_std': [0.229, 0.224, 0.225],
                      'seed': SEED, 'prototype_checkpoint': checkpoint_path.name,
                      'final_test_used': False, 'prototype_only': True}
        (bundle_dir / 'model_spec.json').write_text(json.dumps(model_spec, ensure_ascii=False, indent=2), encoding='utf-8')
        required_files = ['preprocessing.json', 'preprocessing_source.py', 'run_config.json', 'font_splits.json',
            'prepared_cache_config.json', 'font_render_check.json', 'prototype_validation.json', 'prototype_manifest.csv',
            'validation_predictions.csv', 'validation_classification_printed.csv', 'validation_classification_handwritten.csv']
        missing = [name for name in required_files if not (RUN_DIR / name).exists()]
        if missing:
            raise RuntimeError(f'Model export is incomplete; restore current-run metadata before exporting: {missing}')
        if json.loads((RUN_DIR / 'run_config.json').read_text())['run_id'] != RUN_ID or PREPARED_RUN_ID != RUN_ID:
            raise RuntimeError('Run metadata is stale. Load the prepared cache for this run before exporting.')
        for filename in required_files:
            shutil.copy2(RUN_DIR / filename, bundle_dir / filename)
        (bundle_dir / 'source_validation_metrics.json').write_text(json.dumps(source_metrics, ensure_ascii=False, indent=2), encoding='utf-8')
        members = {}
        for item in sorted(bundle_dir.iterdir()):
            if item.is_file() and item.name != 'bundle_manifest.json':
                members[item.name] = sha256_file(item)
        (bundle_dir / 'bundle_manifest.json').write_text(json.dumps({'files_sha256': members}, indent=2), encoding='utf-8')
        archive_path = RUN_DIR / 'prototype_model_bundle.zip'
        shutil.make_archive(str(archive_path.with_suffix('')), 'zip', root_dir=bundle_dir)
        print('Portable prototype bundle:', archive_path, '| bytes:', archive_path.stat().st_size)
        return archive_path

In [ ]:
# BLOCK 13 — Start enabled training and export results
if EXPERIMENT_MODE == 'prepare':
    print('Preparation mode only. No model or final test split is loaded.')
elif EXPERIMENT_MODE == 'prototype':
    if 'PROTOTYPE_READY' not in globals():
        raise RuntimeError('Run the training-manifest/readiness gate cell above before the prototype gate.')
    if not PROTOTYPE_READY:
        raise RuntimeError('Prototype data gate failed. Inspect missing train/validation classes above.')
    print('Prototype mode is available; handwritten آ remains unvalidated.')
    if RUN_PROTOTYPE_TRAINING:
        if not globals().get('PREPARED_DATA_READY', False) or globals().get('PREPARED_RUN_ID') != RUN_ID:
            raise RuntimeError('Run the saved prepared-dataset cell for the current run before training.')
        prototype_manifest = cached_prototype_manifest.copy()
        prototype_manifest.to_csv(RUN_DIR / 'prototype_manifest.csv', index=False)
        loaders, subsets, open_datasets = build_loaders(prototype_manifest, batch_size=16, include_test=False)
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model, best_path, best_score = fit_one_seed(42, loaders, device, prototype=True)
        source_results, selected_score = evaluate_sources(model, loaders['val_by_source'], device)
        validation_summary = {'seed': SEED, 'selection_score': selected_score,
            'source_metrics': source_results, 'checkpoint': str(best_path), 'final_test_used': False,
            'scope': 'prototype validation only; the final test split was not loaded'}
        (RUN_DIR / 'prototype_validation.json').write_text(json.dumps(validation_summary, ensure_ascii=False, indent=2), encoding='utf-8')
        prediction_frame, preview_paths = export_validation_predictions(model, loaders['val_by_source'], device)
        bundle_path = export_prototype_bundle(model, best_path, source_results, prediction_frame)
        print('Prototype validation only:', json.dumps(source_results, ensure_ascii=False, indent=2))
        print('Checkpoint:', best_path)
        print('Prediction table:', RUN_DIR / 'validation_predictions.csv')
        print('Preview images:', {source: str(path) for source, path in preview_paths.items()})
        print('Portable bundle:', bundle_path)
        for dataset in open_datasets.values(): dataset.close()
        for _, _, dataset in loaders['val_by_source'].values(): dataset.close()
    else:
        print('Set RUN_PROTOTYPE_TRAINING = True to start the small Phase 3 run.')
elif EXPERIMENT_MODE == 'final':
    if not FINAL_READY:
        print('Final training locked. It requires reader approval, all printed labels in every split, handwritten آ in every writer split, and at least 12 independent font design groups.')
    elif RUN_FINAL_TRAINING:
        # Test-image preparation occurs only inside the explicitly enabled, reviewed final run.
        final_manifest, final_cache_info = ensure_prepared_cache(
            training_manifest, CACHE_ROOT, RUNTIME_CACHE_ROOT, PREPROCESSING, PREPROCESSING_SOURCE,
            sha256_file(CHARACTER_MAP), SOURCE_MANIFEST_SHA256, EXPECTED_ARCHIVE_SHA256,
            font_splits, source_config, prepare_image, ensure_local_archive, scope='final', allow_test=True)
        final_loaders, final_subsets, final_datasets = build_loaders(final_manifest, batch_size=16, include_test=True)
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        results = []
        for seed in [7, 21, 42]:
            model, best_path, best_score = fit_one_seed(seed, final_loaders, device, prototype=False)
            y, p, _ = predict_loader(model, final_loaders['test'], device)
            by_source = {}
            for source in sorted(final_subsets['test']['source_type'].unique()):
                source_frame = final_subsets['test'][final_subsets['test']['source_type'] == source].reset_index(drop=True)
                source_ds = CharacterDataset(source_frame, LOCAL_ARCHIVE, training=False)
                source_loader = DataLoader(source_ds, batch_size=32, shuffle=False, num_workers=0)
                source_y, source_p, _ = predict_loader(model, source_loader, device)
                supported = sorted(set(source_y.tolist()))
                pd.DataFrame(classification_report(source_y, source_p, labels=list(range(33)),
                    target_names=labels['class_order'], output_dict=True, zero_division=0)).T.to_csv(
                    RUN_DIR / f'classification_seed_{seed}_{source}.csv')
                by_source[source] = {'samples': int(len(source_y)), 'supported_classes': len(supported),
                    'accuracy': float(accuracy_score(source_y, source_p)),
                    'macro_f1_supported': float(f1_score(source_y, source_p, average='macro', labels=supported, zero_division=0))}
                source_ds.close()
            result = {'seed': seed, 'test_samples': int(len(y)), 'test_accuracy': float(accuracy_score(y, p)),
                      'test_macro_f1_supported': float(f1_score(y, p, average='macro', labels=sorted(set(y.tolist())), zero_division=0)),
                      'test_by_source': by_source, 'validation_selection_score': best_score, 'checkpoint': str(best_path)}
            results.append(result)
            pd.DataFrame(classification_report(y, p, labels=list(range(33)), target_names=labels['class_order'],
                output_dict=True, zero_division=0)).T.to_csv(RUN_DIR / f'classification_seed_{seed}_combined.csv')
            print(result)
        (RUN_DIR / 'final_test_summary.json').write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')
        for dataset in final_datasets.values(): dataset.close()
        for _, _, dataset in final_loaders['val_by_source'].values(): dataset.close()
    else:
        print('Final data gate is ready. Set RUN_FINAL_TRAINING = True only after the method and review are complete.')
else:
    raise ValueError('EXPERIMENT_MODE must be prepare, prototype, or final.')

## Output and phase checklist

Preparation data and model runs are independent. The persistent prepared dataset contains 224×224 grayscale images, original sample IDs/splits, a relative-path manifest, exact preprocessing source/settings, font groups, source configuration, and SHA-256 checks. Its completion marker is written last. The selected prototype contains train/validation only; the final test remains reserved.

Each training run references that cache and saves its own model, source-specific validation metrics, reports, predictions, and complete model bundle. Cache hits skip image auditing, font generation, and deterministic image preparation. Training still applies random augmentation, tensor conversion, and normalization to each batch. Reader review stays pending and handwritten `آ` stays unsupported until reviewed samples are integrated.